In [1]:
import pyemu
import os
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning) 
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import shutil
import numpy as np
import sys
import swatmf
import matplotlib.pyplot as plt

In [2]:
swatmf.__version__

'1.0.1'

# 01. Set working directory

In [ ]:
# path to project directory
prj_dir = "C:\\Users\\seonggpa\\Documents\\projects\\watersheds\\hbasnet_opt"
main_opt_path = os.path.join(prj_dir, 'main_opt')
os.chdir(main_opt_path)

# 02. Create prior pst

In [4]:
# create prior pst based on reweigted control file
pst = pyemu.Pst('mb_pp_rw.pst')

In [13]:
pst_prior = 'mb_pp_rw_prior.pst'

In [ ]:
# set prior UA
pst.pestpp_options['ies_drop_conflicts'] = True
pst.pestpp_options['ies_no_noise'] = True
pst.pestpp_options['ies_num_reals'] = 300 # number of realization
pst.control_data.noptmax = -1 # number of iteration
pst.model_command = 'python forward_run.py'
pst.write(f'{pst_prior}', version=2) # write new IES control file

noptmax:-1, npar_adj:83, nnz_obs:5114


In [8]:
os.chdir(os.pardir)

In [9]:
os.getcwd()

'C:\\Users\\seonggpa\\Documents\\projects\\watersheds\\hbasnet_opt'

# 04. Run IES

## 04.1 Set up IES

In [10]:
# check number of cores on your computer
num_workers = psutil.cpu_count(logical=False)

In [ ]:
# name for prior ua directory
m_d = os.path.join(prj_dir, "mb_pp_rw_prior")

## 04.2 Execute

In [ ]:
pyemu.os_utils.start_workers(main_opt_path, # the folder which contains the "template" PEST dataset
                            'pestpp-ies', #the PEST software version we want to run
                            f'{pst_prior}', # the control file to use with PEST
                            num_workers=num_workers, #how many agents to deploy
                            worker_root='.', #where to deploy the agent directories; relative to where python is running
                            master_dir=m_d, #the manager directory,
                            # reuse_master=True
                            )

# 05. Analyze results
## 05.1 Check model performance

In [15]:
pst = pyemu.Pst(os.path.join(m_d, f'{pst_prior}')) # load control file

In [16]:
pst_prior

'mb_pp_rw_prior.pst'

In [17]:
# load prior simulation
pr_oe = pyemu.ObservationEnsemble.from_csv(
    pst=pst,filename=os.path.join(m_d,"mb_pp_rw_prior.0.obs.csv")
    )
# load posterior simulation
# pt_oe = pyemu.ObservationEnsemble.from_csv(pst=pst,filename=os.path.join(m_d,"mb_pp_rw_posterior.{0}.obs.csv".format(pst.control_data.noptmax)))


In [22]:
prior_df = pyemu.ParameterEnsemble.from_csv(pst=pst,filename=os.path.join(m_d,"mb_pp_rw_prior.{0}.par.csv".format(0)))

In [18]:
df_pars = pd.read_csv(os.path.join(m_d, "mb_pp_rw_prior.par_data.csv"))
sel_pars = df_pars.loc[df_pars["partrans"]=='log']
sel_pars

,parnme,partrans,parchglim,parval1,parlbnd,parubnd,pargp,scale,offset,dercom,i,j,zone
0,adj_pkr,log,factor,1.00,0.1000,1.9,swat,1.0,-1.0,1,NaN,NaN,NaN
1,c_factor,log,factor,1.00,0.1000,1.9,swat,1.0,-1.0,1,NaN,NaN,NaN
2,canmx,log,factor,1.00,0.1000,1.9,swat,1.0,-1.0,1,NaN,NaN,NaN
3,ch_k1,log,factor,1.00,0.1000,1.9,swat,1.0,-1.0,1,NaN,NaN,NaN
4,ch_n1,log,factor,1.00,0.1000,1.9,swat,1.0,-1.0,1,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,sy0_i:65_j:55_zone:1,log,factor,0.01,0.0001,0.6,sy,1.0,0.0,1,65.0,55.0,1.0
110,sy0_i:75_j:55_zone:1,log,factor,0.01,0.0001,0.6,sy,1.0,0.0,1,75.0,55.0,1.0
111,sy0_i:85_j:45_zone:1,log,factor,0.01,0.0001,0.6,sy,1.0,0.0,1,85.0,45.0,1.0
112,sy0_i:85_j:55_zone:1,log,factor,0.01,0.0001,0.6,sy,1.0,0.0,1,85.0,55.0,1.0


In [19]:
from swatmf import analyzer

In [25]:
analyzer.plot_prior_posterior_par_hist(m_d, pst_prior, prior_df,prior_df, sel_pars)

AttributeError: 'str' object has no attribute 'parameter_data'